# Chatbot using Hugging Face Transformers

## NLP Task 3

### Objective:
To build a conversational chatbot using a pre-trained transformer model (DialoGPT) that can generate human-like responses.

### Technologies Used:
- Python
- Hugging Face Transformers
- PyTorch
- Google Colab

# 1. Environment Setup

## 1.1 Install Required Libraries

In [1]:
!pip install transformers
!pip install torch

## 1.2 Import Libraries

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# 2. Load Pre-trained Transformer Model

## 2.1 Load Tokenizer & Model

In [3]:
model_name = "microsoft/DialoGPT-medium"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load model
model = AutoModelForCausalLM.from_pretrained(model_name)

print("Model and Tokenizer loaded successfully")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/863M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-medium
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model and Tokenizer loaded successfully


### Model Loading

We are using **DialoGPT-medium**, a pre-trained conversational model from Microsoft.

- It is trained on large-scale conversational data
- Designed specifically for chatbot applications
- Helps generate human-like responses dynamically

## 2.3 Quick Test

In [4]:
# Test encoding
text = "Hello"
input_ids = tokenizer.encode(text + tokenizer.eos_token, return_tensors='pt')

print("Tokenized Input:", input_ids)

Tokenized Input: tensor([[15496, 50256]])


# 3. Response Generation

## 3.1 Basic Response Generator

In [5]:
chat_history_ids = None  # Global variable to store conversation history

def generate_response(user_input):
    global chat_history_ids

    # Encode user input
    new_input_ids = tokenizer.encode(user_input + tokenizer.eos_token, return_tensors='pt')

    # Create attention mask
    attention_mask = torch.ones(new_input_ids.shape, dtype=torch.long)

    # Combine with previous chat history
    bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1) if chat_history_ids is not None else new_input_ids

    # Generate response from model
    chat_history_ids = model.generate(
        bot_input_ids,
        attention_mask=torch.ones(bot_input_ids.shape, dtype=torch.long),
        max_length=1000,
        pad_token_id=tokenizer.eos_token_id
    )

    # Decode only the newly generated response
    response = tokenizer.decode(
        chat_history_ids[:, bot_input_ids.shape[-1]:][0],
        skip_special_tokens=True
    )

    return response

## 3.2 Test the Function

In [6]:
print(generate_response("Hello"))

Hello! :D


## Response Generation

The chatbot generates responses using a transformer model:

- User input is converted into tokens
- Tokens are passed to the DialoGPT model
- The model generates a response based on context
- Previous conversation is stored for better interaction

This ensures dynamic and human-like responses.

# 4. Continuous Chatbot Loop

## 4.1 Chat Loop Code

In [8]:
print("Chatbot: Hello! I am your AI assistant. Type 'exit' or 'quit' to stop.\n")

chat_history_ids = None

while True:
    user_input = input("You: ")

    if user_input.lower() in ["exit", "quit"]:
        print("Chatbot: Goodbye! Have a great day")
        break

    # Add instruction to improve response quality
    enhanced_input = "Answer clearly and professionally: " + user_input

    new_input_ids = tokenizer.encode(enhanced_input + tokenizer.eos_token, return_tensors='pt')

    bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1) if chat_history_ids is not None else new_input_ids

    attention_mask = torch.ones(bot_input_ids.shape, dtype=torch.long)

    chat_history_ids = model.generate(
        bot_input_ids,
        attention_mask=attention_mask,
        max_length=1000,

        # KEY IMPROVEMENTS
        do_sample=True,
        top_k=50,
        top_p=0.92,
        temperature=0.7,
        no_repeat_ngram_size=3,

        pad_token_id=tokenizer.eos_token_id
    )

    response = tokenizer.decode(
        chat_history_ids[:, bot_input_ids.shape[-1]:][0],
        skip_special_tokens=True
    )

    print("Chatbot:", response)

Chatbot: Hello! I am your AI assistant. Type 'exit' or 'quit' to stop.

You: exit
Chatbot: Goodbye! Have a great day


## Continuous Conversation

The chatbot runs in a loop to simulate real-time interaction:

- Accepts user input continuously
- Maintains conversation history
- Generates context-aware responses
- Stops when the user types "exit" or "quit"